# General Block Forward Example

Define a block-structured general problem with `HighLevelNLPBuilder`, generate labels with SLSQP, then save and train through `ProblemBuilder.run(...)`.


## Setup
Import JAX, NumPy, the block builder, label-generation helpers, and the builder-owned runner.


In [1]:
from argparse import Namespace
from pathlib import Path
import json
import time

import jax.numpy as jnp
import numpy as np

from jaxmodel import HighLevelNLPBuilder
from kkthn.builder import ProblemBuilder
from kkthn.problems import apply_label_noise, generate_labels_with_slsqp


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def make_output_root(name):
    output_root = ROOT / "notebooks" / "_runs" / name
    output_root.mkdir(parents=True, exist_ok=True)
    return output_root


def latest_run_dir(output_root):
    runs = [path for path in output_root.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found under {output_root}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def load_summary(run_dir):
    with open(run_dir / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)


## Configuration
Edit these dictionaries to change the sample count, parameter box, training loop, or projection settings.


In [2]:
DATA = {
    "type": "general_block",
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "x_L": [-1.0, -1.0],
    "x_U": [1.0, 1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


## Problem Definition
This block-style model is useful when constraints are naturally written as vectorized JAX functions.


In [3]:
def build_block_problem(dtype=jnp.float64):
    n_x = 2
    n_y = 4
    q_diag = jnp.asarray([1.4, 1.2, 0.8, 0.9], dtype=dtype)

    def objective(params, vars_dict):
        del params
        y = vars_dict["y"]
        return 0.5 * jnp.sum(q_diag * y**2) + 0.05 * jnp.sum(jnp.sin(y))

    def equality_block(params, vars_dict):
        x = params["x"]
        y = vars_dict["y"]
        return jnp.asarray([
            y[0] + 0.10 * y[2] ** 2 - x[0],
            y[1] + 0.10 * y[3] ** 2 - x[1],
        ], dtype=dtype)

    def inequality_block(params, vars_dict):
        del params
        y = vars_dict["y"]
        return jnp.asarray([
            y[0] ** 2 + y[2] ** 2 - 3.0,
            y[1] ** 2 + y[3] ** 2 - 3.0,
        ], dtype=dtype)

    zeros = jnp.zeros((n_y, n_x), dtype=dtype)
    lower = -3.0 * jnp.ones((n_y,), dtype=dtype)
    upper = 3.0 * jnp.ones((n_y,), dtype=dtype)
    params0 = {"x": jnp.zeros((n_x,), dtype=dtype)}

    return (
        HighLevelNLPBuilder(dtype=dtype)
        .add_parameter("x", n_x)
        .add_variable("y", n_y)
        .set_objective(objective)
        .add_nonlinear_equality(equality_block, name="equality_block")
        .add_nonlinear_inequality(inequality_block, name="inequality_block")
        .set_affine_lower_bound(var_name="y", param_name="x", M=zeros, c=lower)
        .set_affine_upper_bound(var_name="y", param_name="x", M=zeros, c=upper)
        .build(example_params=params0, jit_compile=True)
    )


## Train
Sample parameters, generate labels with SLSQP, then call `ProblemBuilder.run(...)` with the prepared arrays.


In [4]:
output_root = make_output_root("general_block_forward")
args = Namespace(type="general_block", action="run", mode="forward", output_dir=str(output_root))
model = build_block_problem()

rng = np.random.default_rng(DATA["seed"])
X = rng.uniform(DATA["x_L"], DATA["x_U"], size=(DATA["num_samples"], 2))
Y, label_metadata = generate_labels_with_slsqp(model, X)
Y, label_metadata = apply_label_noise(Y, DATA, label_metadata)
label_metadata.update({
    "problem_type": "general_block",
    "n_x": 2,
    "n_y": 4,
    "n_eq": 2,
    "n_ineq": 2,
})

exit_code = ProblemBuilder.run(
    args,
    root=ROOT,
    model=model,
    X=X,
    Y=Y,
    data=DATA,
    train=CONFIG,
    proj=PROJ,
    metadata=label_metadata,
    parameter_names=["x1", "x2"],
    variable_names=["y1", "y2", "y3", "y4"],
)
run_dir = latest_run_dir(output_root)
summary_json = load_summary(run_dir)

print(f"Exit code: {exit_code}")
print(f"Run directory: {run_dir}")
print(f"Saved artifacts: {summary_json['artifacts']}")


KKT-HardNet
  dims: n_x=2 n_y=4 n_eq=2 n_ineq=10
  mode: forward
  samples: train=9 val=3 batch_size=4
  network: [2, 32, 32, 4]
epoch 0001 | train=1.508205e-02 val=1.537898e-02 raw_val=5.178504e-01 eq=1.028e-11 ineq=0.000e+00 | train_batch=1.1927s val_batch=0.0014s
epoch 0002 | train=7.887839e-03 val=1.020399e-02 raw_val=5.145874e-01 eq=8.540e-12 ineq=0.000e+00 | train_batch=0.0013s val_batch=0.0014s
epoch 0003 | train=4.157275e-03 val=6.768309e-03 raw_val=5.111071e-01 eq=7.393e-12 ineq=0.000e+00 | train_batch=0.0012s val_batch=0.0012s
Saved run artifacts: D:\Projects\KKTHardNet\notebooks\_runs\general_block_forward\general_block_forward_20260414_121828
Training time: 8.278s
=== Timing summary ===
Train step time total      : 3.581s
Validation time total      : 0.004s
Avg train time / epoch     : 1.1935s
Avg validation time / epoch: 0.0013s
Avg train time / batch     : 0.3978s
Avg validation time / batch: 0.0013s
=== Profiled component time distribution ===
Backbone forward :   0.03%


## Summary
Read the emitted `summary.json` from the newest run folder.


In [5]:
summary = {
    "output_dir": str(run_dir),
    "dims": summary_json["dims"],
    "final": summary_json["final"],
    "component_time_percent": summary_json["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))


{
  "output_dir": "D:\\Projects\\KKTHardNet\\notebooks\\_runs\\general_block_forward\\general_block_forward_20260414_121828",
  "dims": {
    "n_eq": 2,
    "n_ineq": 10,
    "n_x": 2,
    "n_y": 4,
    "n_z": 26,
    "raw_n_ineq": 2
  },
  "final": {
    "epoch": 3,
    "mean_batch_loss": 0.006150899562874252,
    "train_batches": 3,
    "train_epoch_time_sec": 0.0035410999999996307,
    "train_eq_l2": 5.021485458167505e-12,
    "train_eval_time_sec": 0.0008173000000013531,
    "train_ineq_l2": 0.0,
    "train_loss": 0.004157274539130844,
    "train_raw_mse": 0.6254943001069075,
    "train_step_time_per_batch_sec": 0.0007014000000017025,
    "train_step_time_sec": 0.0021042000000051075,
    "train_time_per_batch_sec": 0.0011803666666665436,
    "val_eq_l2": 7.392807762195245e-12,
    "val_ineq_l2": 0.0,
    "val_loss": 0.006768308563433893,
    "val_raw_mse": 0.5111071045557141,
    "validation_batches": 1,
    "validation_epoch_time_sec": 0.0011630999999994174,
    "validation_time_p